# Instructions
Ensure that you have added all assemblies and parts.
All relevant assemblies should have associates sub-assembly parts added, or this will throw errors

If there is an error, it is setup to fail 

In [12]:
# Load data from the spreadsheet
import sys
import pandas as pd
from pathlib import Path
import openpyxl

from loguru import logger
logger.remove()
logger.add(sys.stderr, level="ERROR")

filename = Path(r"C:\Users\crdig\Downloads\ENOSCEN_APMay 08 2025 15_43_30.csv")
assert filename.exists(), f"File {filename} does not exist."

# Import the data
if filename.suffix == ".csv":
    df_raw = pd.read_csv(filename, header=0, dtype=object)
elif filename.suffix == ".xlsx":
    df_raw = pd.read_excel(filename, sheet_name="Sheet1", skiprows=0, usecols="A:EL", header=None, dtype=object)
# df_raw

In [13]:
# Categorize the data

# Iterate through each row. Create a new column "Assembly" and set it to 0 if part, 1 if assembly.
#   If the next row has a higher number, than the current line is an assembly.
#   If the next row has the same number, than the current line is a part.
#   If the next row has a lower number, than the current line is a part.
df_tagged = df_raw.copy()
df_tagged["Assembly"] = 0
for i in range(len(df_tagged) - 1):
    if df_tagged.iloc[i, 0] < df_tagged.iloc[i + 1, 0]:
        df_tagged.iloc[i, -1] = 1
    elif df_tagged.iloc[i, 0] == df_tagged.iloc[i + 1, 0]:
        df_tagged.iloc[i, -1] = 0
    else:
        df_tagged.iloc[i, -1] = 0

df_assemblies = df_tagged[df_tagged["Assembly"] == 1].copy()
df_parts = df_tagged[df_tagged["Assembly"] == 0].copy()

# Remove duplicates from the assemblies and parts
df_assemblies = df_assemblies.drop_duplicates(subset=['Title'], keep="first")
df_assemblies.reset_index(drop=True, inplace=True)
df_assemblies.sort_values(by=["Title"], inplace=True)

df_parts = df_parts.drop_duplicates(subset=['Title'], keep="first")
df_parts.reset_index(drop=True, inplace=True)
df_parts.sort_values(by=["Title"], inplace=True)

print(f"{len(df_assemblies)} assemblies found.")
print(f"{len(df_parts)} parts found.")

# df_tagged
# df_assemblies
# df_parts

7 assemblies found.
26 parts found.


In [14]:
# Init PartsBoxAPI

# Add parts to PartsBox and then to assemblies
import sys
from PartsBoxAPI.PartsBoxAPI import PartsBoxAPI
from datetime import datetime
# Add all non-existing parts into partsbox
tags = ["3DExperience"]  # You need to create the tag in PartsBox first for assemblies and parts separately.

logger.remove()
logger.add(sys.stderr, level="ERROR")

# Actual API
link = "https://partsbox.com/pcnzl"
PartsBox = PartsBoxAPI("partsboxapi_6cf6evkbnmhr4a70pqxzd2hcvab3bda5c95114707b905ab70858635cb5cd549d")  # Real API key

# Test API
# link = "https://partsbox.com/chlepcnzl"
# PartsBox = PartsBoxAPI("partsboxapi_8dvrrisdiggkmg1tekt3973dpga6c2exee90d5a4d82ed8964decb742a7ff518c85d9326")  # Test API key


In [15]:
# Create a linked tree of parts to allow for easy BOM creation

# Top level tree -> {Name:[Quantity, {Subitems...}, Name:[Quantity, {Subitems...}]}
# Subitems follows top level structure

def build_bom_recursive(df, start_index=0, current_level=0):
    bom = {}
    i = start_index
    n = len(df)

    while i < n:
        row = df.iloc[i]
        level = int(row["Level"])
        title = row["Title"]

        if level < current_level:
            # Return control to the previous level
            return bom, i

        elif level > current_level:
            # Should not happen — nested levels must come after a parent
            # But we call recursively from parent, so it's safe to ignore here
            raise ValueError(f"Unexpected level jump at row {i}: {title}")

        # If the next row is a child, recursively build its sub-BOM
        next_index = i + 1
        if next_index < n:
            next_level = int(df.iloc[next_index]["Level"])
        else:
            next_level = -1  # signals end of list

        if next_level > level:
            # Next item is a child — build its sub-bom
            sub_bom, consumed_index = build_bom_recursive(df, i + 1, current_level + 1)
            if title in bom:
                bom[title][0] += 1  # increment quantity
            else:
                bom[title] = [1, sub_bom]
            i = consumed_index
        else:
            # Leaf item (or sibling)
            if title in bom:
                bom[title][0] += 1
            else:
                bom[title] = [1, None]
            i += 1

    return bom, i

# Flatten the tree into a list of single level BOMs
# [{Asssembly}:{Part:Quantity. Part:Quantity, ...}, ...]
flat_bom = {}
def flatten_bom(bom):
    global flat_bom
    
    for current_assembly in bom.items():
        assembly_name = current_assembly[0]
        subitems = current_assembly[1][1]
        if subitems is not None:  # If the current item is an assembly
            flatten_bom(subitems)
            if assembly_name not in flat_bom.keys():  # And hasn't already been listed
                flat_bom[assembly_name] = {}  # Create the entry
                for subitem in subitems.items():  # For each subitem, add the item
                    subitem_name, subitem_quantity = subitem[0], subitem[1][0]
                    flat_bom[assembly_name][subitem_name] = subitem_quantity
                    
    return

root = build_bom_recursive(df_raw)[0]
flatten_bom(root)

# root
# flat_bom

In [16]:
# Add the BOMs to each assembly

# Get all assemblies from PartsBox
assemblies = PartsBox.projects.get_all_projects()['data']
assemblies = {assembly["project/name"]: assembly for assembly in assemblies}
# Get all parts and assemblies from PartsBox
parts = PartsBox.parts.get_all_parts()['data']
parts = {part["part/name"]: part for part in parts}

items_to_add = {}
items_to_update = {}
for current in flat_bom.keys():
    part_counter = 0
    current_assembly = assemblies[current]
    if current_assembly["project/id"] in items_to_add.keys():
        continue
    project_entries = PartsBox.projects.get_project_entries(current_assembly["project/id"])['data']
    
    items_to_add[current_assembly["project/id"]] = []
    items_to_update[current_assembly["project/id"]] = []
    for part in flat_bom[current].items():
        entry = {
            "entry/part-id": parts[part[0]]["part/id"],
            "entry/quantity": part[1],
            "entry/name": part[0],
            "entry/designators": [f"P{i + part_counter}" for i in range(part[1])]
        }
        part_counter += part[1]
        items_to_add[current_assembly["project/id"]].append(entry)
        entry_list = {entry["entry/part-id"]:entry["entry/id"] for entry in project_entries}
        if entry["entry/part-id"] in entry_list.keys():
            items_to_update[current_assembly["project/id"]].append(entry_list[entry["entry/part-id"]])
        
        print(f"{current_assembly['project/name']}: Found {len(items_to_add[current_assembly['project/id']])} items to add and {len(items_to_update[current_assembly['project/id']])} to update.")

print(items_to_add)
print(items_to_update)

PRT0013-01: Found 1 items to add and 1 to update.
PRT0013-01: Found 2 items to add and 1 to update.
PRT0011-01: Found 1 items to add and 1 to update.
PRT0011-01: Found 2 items to add and 2 to update.
PRT0011-01: Found 3 items to add and 3 to update.
PRT0011-01: Found 4 items to add and 4 to update.
PRT0019-01: Found 1 items to add and 1 to update.
PRT0019-01: Found 2 items to add and 2 to update.
PRT0028-01: Found 1 items to add and 1 to update.
PRT0028-01: Found 2 items to add and 2 to update.
PRT0028-01: Found 3 items to add and 3 to update.
PRT0032-01: Found 1 items to add and 1 to update.
PRT0032-01: Found 2 items to add and 2 to update.
PRT0035-01: Found 1 items to add and 1 to update.
PRT0035-01: Found 2 items to add and 2 to update.
HLA0001-01: Found 1 items to add and 1 to update.
HLA0001-01: Found 2 items to add and 2 to update.
HLA0001-01: Found 3 items to add and 3 to update.
HLA0001-01: Found 4 items to add and 4 to update.
HLA0001-01: Found 5 items to add and 5 to update.


In [17]:
# Upload results  
for project_id in items_to_add.keys(): 
    if len(items_to_update[project_id]) > 0:
        PartsBox.projects.delete_project_entries(
            project_id=project_id,
            ids=items_to_update[project_id],
        )
    if len(items_to_add[project_id]) > 0:
        PartsBox.projects.add_project_entries(
            project_id=project_id,
            entries=items_to_add[project_id],
        )